# Text encoder (ModernBERT) on Google Colab

Runs the `encoder` model through the shared pipeline on a Colab GPU (see `docs/TEAM_GUIDE.md`).

**Before you start:** Runtime → Change runtime type → **A100 GPU** (or L4). Keep this tab open while it trains.

Everything that must survive the session lives in Google Drive, in `MyDrive/llmrouter_encoder/`:
the built data, the results folders and `logs/` (the record of data reads, including test reads).
If Colab disconnects, run cells 1–6 again, then the cell you were on: the development run resumes
from the last finished rotation.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['WORK'] = '/content/drive/MyDrive/llmrouter_encoder'   # survives the session
os.environ['REPO'] = '/content/llm-routing-study'                 # fast local disk, wiped with the session
os.makedirs(os.environ['WORK'], exist_ok=True)

## 3. Code (`encoder` branch)

In [ ]:
%%bash
set -e
if [ -d "$REPO/.git" ]; then
  cd "$REPO" && git fetch -q && git checkout -q encoder && git pull -q
else
  git clone -q -b encoder https://github.com/rahulkanagaraj786/llm-routing-study.git "$REPO"
fi
cd "$REPO" && git log --oneline -1

## 4. Packages

Installs `requirements.txt`. If the pinned torch build cannot use this machine's GPU driver,
it falls back to torch 2.14.1 for CUDA 12.6 (the version actually used is recorded in every result file).

In [ ]:
%%bash
cd "$REPO"
pip install -q -r requirements.txt
if ! python -c "import torch, sys; sys.exit(0 if torch.cuda.is_available() else 1)" 2>/dev/null; then
  echo "The pinned torch build cannot use this GPU driver; installing torch 2.14.1 for CUDA 12.6"
  pip install -q torch==2.14.1 --index-url https://download.pytorch.org/whl/cu126
fi
python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU:', torch.cuda.get_device_name(0), '| native bf16:', torch.cuda.is_bf16_supported(including_emulation=False))"

## 5. Data (built once, then copied from Drive)

In [ ]:
%%bash
set -e
cd "$REPO"
mkdir -p data/processed
if [ -f "$WORK/data/processed/paired_v2.jsonl.gz" ]; then
  cp "$WORK"/data/processed/* data/processed/
  echo "data copied from Drive"
else
  mkdir -p data/raw/extracted
  curl -sSL -o data/raw/bench-release.tar.gz "https://huggingface.co/datasets/NPULH/LLMRouterBench/resolve/0e5af1b84bf73437a01a1849c0f1d2468baa93fc/bench-release.tar.gz"
  echo "b79f8cde1a6f029c2efa663a3a3b6f7748defb22341fe59f328cebef6648c8f1  data/raw/bench-release.tar.gz" | sha256sum -c
  tar -xzf data/raw/bench-release.tar.gz -C data/raw/extracted -T data/selected_members.txt
  python scripts/01_build_paired.py | tail -3
  mkdir -p "$WORK/data/processed" && cp data/processed/* "$WORK/data/processed/"
  rm -rf data/raw
  echo "data built and saved to Drive"
fi

## 6. Keep results and logs on Drive

In [ ]:
%%bash
set -e
cd "$REPO"
for d in results/dev_selection/encoder results/final_dryrun/encoder results/final/encoder logs; do
  mkdir -p "$WORK/$d" "$(dirname "$d")"
  if [ -d "$d" ] && [ ! -L "$d" ]; then cp -rn "$d"/. "$WORK/$d"/ && rm -rf "$d"; fi   # keep anything already written
  [ -L "$d" ] || ln -s "$WORK/$d" "$d"
  echo "$d -> $(readlink "$d")"
done

## 7. Tests and smoke check

Check that the epoch lines end with `cuda` and that it prints `SMOKE OK`. Nothing is written.

In [ ]:
%%bash
cd "$REPO"
python tests/test_policy.py && python tests/test_guards.py && python tests/test_models.py
python scripts/05_dev_compare.py encoder --smoke

## 8. Development run (six fine-tunes, resumable)

Output: `MyDrive/llmrouter_encoder/results/dev_selection/encoder/`. When it finishes, copy that folder
into `results/dev_selection/encoder/` in the repository and commit it **before** step 10.

In [ ]:
%%bash
cd "$REPO"
python scripts/05_dev_compare.py encoder 2>&1 | tee -a "$WORK/dev_encoder.log"

## 9. Rehearsal of the final evaluation (development questions only)

In [ ]:
%%bash
cd "$REPO"
python scripts/07_final_eval.py encoder --plan
python scripts/07_final_eval.py encoder --dry-run 2>&1 | tee -a "$WORK/dryrun_encoder.log"
python scripts/09_report_final.py encoder --dry-run

## 10. Final evaluation: the RQ1 test, ONCE

Only after the development results are committed and **the team has agreed**. It reads the locked
test split and refuses to run a second time. Never delete or rename the results to run it again.
Afterwards copy `results/final/encoder/` (without the `model/` folder) into the repository and commit it.

In [ ]:
TEAM_AGREED = False   # set to True only after the team agrees to run the final evaluation

import subprocess
if not TEAM_AGREED:
    print("Not run. Set TEAM_AGREED = True once the team has agreed.")
else:
    subprocess.run('cd "$REPO" && python scripts/07_final_eval.py encoder --unlock-final 2>&1 | tee -a "$WORK/final_encoder.log" '
                   '&& python scripts/09_report_final.py encoder', shell=True, check=True, executable='/bin/bash')